In [0]:
%pip install -q databricks-sdk>=0.118.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Milestone 2.1 — Create Lakebase Project + Dev Branch (idempotent)
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import (
    Project, ProjectSpec, Branch, BranchSpec
)
from databricks.sdk.errors import AlreadyExists, BadRequest, ResourceConflict

w = WorkspaceClient()

PROJECT_ID = "meridian-bank"

# Create project (autoscaling, PG 17) — skip if already exists
try:
    op = w.postgres.create_project(
        project=Project(spec=ProjectSpec(display_name="Meridian Bank", pg_version=17)),
        project_id=PROJECT_ID,
    )
    project = op.wait()
    print(f"Created: {project.name}")
except (AlreadyExists, BadRequest, ResourceConflict):
    print(f"Project '{PROJECT_ID}' already exists — skipping creation.")

# Create dev branch (permanent, copy-on-write from production) — skip if already exists
try:
    w.postgres.create_branch(
        parent=f"projects/{PROJECT_ID}",
        branch=Branch(spec=BranchSpec(
            source_branch=f"projects/{PROJECT_ID}/branches/production",
            no_expiry=True,
        )),
        branch_id="dev",
    ).wait()
    print("Dev branch created.")
except (AlreadyExists, BadRequest, ResourceConflict):
    print("Dev branch already exists — skipping creation.")

# Verify connectivity
print("\nBranches:")
for b in w.postgres.list_branches(parent=f"projects/{PROJECT_ID}"):
    print(f"  {b.name} — state: {b.status.current_state}")
print("\nEndpoints:")
for e in w.postgres.list_endpoints(parent=f"projects/{PROJECT_ID}/branches/production"):
    print(f"  Endpoint host: {e.status.hosts.host}")



Project 'meridian-bank' already exists — skipping creation.
Dev branch already exists — skipping creation.

Branches:
  projects/meridian-bank/branches/forecasting-q3-2025 — state: BranchStatusState.READY
  projects/meridian-bank/branches/production — state: BranchStatusState.READY
  projects/meridian-bank/branches/dev — state: BranchStatusState.READY

Endpoints:
  Endpoint host: ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net
